# 💳 Notebook 3: Payment & Reservation Flow

When a user picks a seat on Ticketmaster and clicks **"Buy"**, they still need to
enter their credit-card details and wait for the payment to process. That takes
**2–10 minutes**. During that time we need a way to **temporarily hold** the seat
so nobody else can snatch it.

This notebook walks through the problem step by step — from a naive approach that
breaks under load, all the way to the production-grade pattern used by real
ticketing platforms: **Redis distributed locks with TTL**.

---

### 🎯 Learning Objectives

By the end of this notebook you will understand:

| # | Concept |
|---|--------|
| 1 | Why temporary holds are needed during checkout |
| 2 | How `SET NX EX` in Redis creates an **atomic distributed lock** |
| 3 | The **reserve → pay → confirm** lifecycle of a booking |
| 4 | How TTL expiration automatically frees abandoned seats |
| 5 | Why **idempotent** payment confirmation prevents double-charges |

> **Prerequisites:** You should have completed *Notebook 1 (Seat Selection)* and
> *Notebook 2 (Flash Sales)* first, or at least be comfortable with basic SQL
> and Redis commands.

## 🛠️ Setup

Start the containers (PostgreSQL, Redis, Adminer, RedisInsight):

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

Then create the virtual environment and install dependencies (first time only):

```bash
uv venv
source .venv/bin/activate          # Windows: .venv\\Scripts\\activate
uv sync
```

### 🔌 Kernel Selection

In VS Code, click the **kernel picker** (top-right of the notebook) and select
the `.venv` kernel. If it doesn't appear, reload the window
(`Cmd+Shift+P` → *"Reload Window"*).

### 🔍 Visualization Tools

| Tool | URL | Notes |
|------|-----|-------|
| **Adminer** (PostgreSQL GUI) | [http://localhost:8081](http://localhost:8081) | System: PostgreSQL, Server: `postgres`, User: `demo`, Password: `demo`, DB: `ticketmaster` |
| **RedisInsight** (Redis GUI) | [http://localhost:5541](http://localhost:5541) | Add database → Host: `host.docker.internal`, Port: `6380` |

> 💡 Keep both tools open in browser tabs — you'll see data change in real time
> as you run cells.

In [ ]:
import psycopg2
import redis
import time
import json
import random
import threading
import uuid
from datetime import datetime
from tabulate import tabulate

# ── Configuration ────────────────────────────────────────
PG_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "ticketmaster",
    "user": "demo",
    "password": "demo",
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6380,
    "decode_responses": True,
}

# ── Helper: get a fresh PostgreSQL connection ────────────
def get_pg_conn(autocommit=True):
    conn = psycopg2.connect(**PG_CONFIG)
    conn.autocommit = autocommit
    return conn

# ── Helper: get a Redis client ─────────────────────
def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# ── Helper: run a SELECT and return rows ───────────────
def query(sql, params=None):
    conn = get_pg_conn()
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return cols, rows

# ── Helper: pretty-print a query result ────────────────
def show(sql, params=None, title=None):
    cols, rows = query(sql, params)
    if title:
        print(f"\n{title}")
    print(tabulate(rows, headers=cols, tablefmt="simple_outline"))

# ── Test connections ───────────────────────────────────
conn = get_pg_conn()
cur = conn.cursor()
cur.execute("SELECT version();")
pg_version = cur.fetchone()[0].split(",")[0]
cur.close()
conn.close()

r = get_redis()
redis_ping = r.ping()

print(f"✅ PostgreSQL connected — {pg_version}")
print(f"✅ Redis connected     — ping={redis_ping}")

## 🤔 The Problem: Payment Takes Time

Think about the user journey when buying a concert ticket:

```
1. 🔍  Browse event  →  pick seats
2. 🛒  Click "Buy"   →  enter credit-card details (1-5 min)
3. 💳  Submit payment →  wait for bank to approve  (5-30 sec)
4. ✅  Confirmation   →  ticket is yours!
```

Steps 2 and 3 take **2–10 minutes**. During that entire window the seat is in
limbo — the user *wants* it but hasn't *paid* for it yet.

### 😱 What happens without a reservation?

```
Timeline
────────────────────────────────────────────────
t=0    User A selects seat FLOOR-A-1
t=0    User B selects seat FLOOR-A-1
t=1    User A starts entering credit-card info …
t=2    User B finishes quickly, clicks Pay ✅
t=5    User A clicks Pay → "Sorry, that seat was already sold!" 😤
```

User A spent **5 minutes** filling out a form for nothing.  That's a terrible
experience.

### 💡 The solution: **temporarily hold** the seat

We need a mechanism that:

1. **Locks** the seat the moment the user clicks "Buy"
2. **Holds** it for a limited time (e.g. 10 minutes)
3. **Releases** it automatically if the user abandons the checkout
4. **Confirms** it permanently once payment succeeds

Let's explore three approaches — from worst to best.

## ❌ Bad Approach: Long-Running Database Locks

The most obvious idea:

```sql
BEGIN;
SELECT * FROM tickets WHERE id = 42 FOR UPDATE;  -- lock the row
-- ... wait 5-10 minutes while user pays ...
UPDATE tickets SET status = 'sold' WHERE id = 42;
COMMIT;
```

Why this is **terrible**:

| Problem | Explanation |
|---------|-------------|
| **Holds a DB connection** | Each checkout keeps a connection open for minutes. With 10,000 concurrent users you'd need 10,000 connections — databases max out around a few hundred. |
| **Deadlock risk** | If two transactions lock rows in different order, PostgreSQL kills one. |
| **Idle-in-transaction** | PostgreSQL's `idle_in_transaction_session_timeout` will terminate you. |
| **No auto-release** | If the app crashes mid-checkout, the lock stays until the connection times out. |

> ⚠️ **Rule of thumb:** never hold a database transaction open for more than a
> few hundred milliseconds. Anything longer should use an external lock.

In [ ]:
# ── Demonstrate the problem with long-running DB locks ──────
# We'll use two connections to show how SELECT FOR UPDATE blocks.

# Pick an available ticket for event 1
cols, rows = query(
    "SELECT id, section, row_label, seat_number FROM tickets "
    "WHERE event_id = 1 AND status = 'available' ORDER BY id LIMIT 1"
)
demo_ticket_id = rows[0][0]
print(f"🎫 Using ticket {demo_ticket_id} ({rows[0][1]}-{rows[0][2]}-{rows[0][3]})")

results = {}

def user_a_holds_lock():
    """User A grabs the lock and holds it for 5 seconds (simulating slow payment)."""
    conn = get_pg_conn(autocommit=False)
    cur = conn.cursor()
    cur.execute("SELECT id FROM tickets WHERE id = %s FOR UPDATE", (demo_ticket_id,))
    results["a_locked"] = time.time()
    print(f"🅰️  User A locked ticket {demo_ticket_id} at t=0.0s")

    time.sleep(5)  # simulate slow payment

    conn.rollback()  # release without buying
    cur.close()
    conn.close()
    results["a_released"] = time.time()
    print(f"🅰️  User A released lock after 5.0s")

def user_b_tries_lock():
    """User B tries to access the same ticket — gets blocked!"""
    time.sleep(0.3)  # small delay so A grabs first
    conn = get_pg_conn(autocommit=False)
    cur = conn.cursor()

    start = time.time()
    results["b_waiting"] = start
    print(f"🅱️  User B trying to lock ticket {demo_ticket_id} …")

    cur.execute("SELECT id FROM tickets WHERE id = %s FOR UPDATE", (demo_ticket_id,))
    wait_time = time.time() - start
    results["b_acquired"] = time.time()
    print(f"🅱️  User B finally got access after waiting {wait_time:.1f}s ⏳")

    conn.rollback()
    cur.close()
    conn.close()

t1 = threading.Thread(target=user_a_holds_lock)
t2 = threading.Thread(target=user_b_tries_lock)
t1.start()
t2.start()
t1.join()
t2.join()

wait = results["b_acquired"] - results["b_waiting"]
print(f"\n⚠️  User B was blocked for {wait:.1f} seconds — imagine this at 10 minutes!")
print("❌ This approach does NOT scale.")

## ⚠️ Better Approach: Status + Expiration in the Database

A smarter idea: add a `reserved_until` column to the tickets table.

```sql
ALTER TABLE tickets
    ADD COLUMN reserved_by    INTEGER,
    ADD COLUMN reserved_until TIMESTAMP;
```

Then the workflow becomes:

```sql
-- 1. Reserve (short transaction)
UPDATE tickets
SET    status = 'reserved',
       reserved_by = 101,
       reserved_until = NOW() + INTERVAL '10 minutes'
WHERE  id = 42 AND status = 'available';

-- 2. Periodic cleanup (cron job every 30 seconds)
UPDATE tickets
SET    status = 'available', reserved_by = NULL, reserved_until = NULL
WHERE  status = 'reserved' AND reserved_until < NOW();
```

### ✅ What's good

- Short database transactions (milliseconds, not minutes)
- Clear status tracking

### ❌ What's still bad

| Problem | Why it matters |
|---------|---------------|
| **Cron lag** | If the cron runs every 30s, a ticket could stay "reserved" for 30s longer than intended. |
| **Clock drift** | Different servers may disagree on "now". |
| **Polling overhead** | The cron query scans all reserved tickets every cycle. |
| **No atomic check-and-set** | Two servers might try to reserve the same ticket simultaneously. |

We can do better with Redis.

## ✅ Best Approach: Redis Distributed Locks

The key insight:

> Tickets only have **two permanent states** in PostgreSQL: `available` and `sold`.
>
> The temporary "reserved" state lives **entirely in Redis** with automatic TTL.

### 🔑 The magic command

```
SET ticket:{id}:lock {userId} NX EX 600
```

| Flag | Meaning |
|------|--------|
| `NX` | **Only set if the key does Not eXist** — atomic "first one wins" |
| `EX 600` | **Expire after 600 seconds** (10 minutes) — auto-cleanup |

This single command is:
- **Atomic** — Redis is single-threaded, no race conditions
- **Self-cleaning** — TTL expires automatically, no cron needed
- **Fast** — Redis handles 100K+ ops/sec

### 🏗️ Architecture

```
Reserve Flow:
┌──────────┐     ┌─────────────────┐     ┌───────┐     ┌────────────┐
│  Client   │────▶│ Booking Service  │────▶│ Redis │────▶│ PostgreSQL │
│ (Buy btn) │     │                 │     │SET NX │     │ INSERT     │
└──────────┘     └─────────────────┘     │EX 600│     │ booking    │
                                          └───────┘     └────────────┘

Confirm Flow (after payment):
┌──────────┐     ┌─────────────────┐     ┌────────────┐     ┌───────┐
│  Stripe   │────▶│ Booking Service  │────▶│ PostgreSQL │────▶│ Redis │
│ webhook  │     │                 │     │ UPDATE     │     │  DEL  │
└──────────┘     └─────────────────┘     │ ticket+bkg │     │  key  │
                                          └────────────┘     └───────┘

Expiry Flow (user abandons):
┌───────┐
│ Redis │──── TTL expires ──── key auto-deleted ──── ticket is available again!
└───────┘
```

Let's implement this!

In [ ]:
# ── Reserve a ticket using Redis distributed lock ────────

def reserve_ticket(event_id, user_id, ticket_id, ttl_seconds=600):
    """
    Attempt to reserve a ticket for a user.

    Steps:
      1. Try to acquire a Redis lock (SET NX EX)
      2. If lock acquired -> create an in-progress booking in PostgreSQL
      3. If lock fails   -> someone else already has it

    Returns:
      booking_id (int) on success, None on failure.
    """
    r = get_redis()
    lock_key = f"ticket:{ticket_id}:lock"

    # Step 1: Atomic lock attempt
    # NX = only set if key doesn't exist (first one wins)
    # EX = auto-expire after ttl_seconds
    acquired = r.set(lock_key, str(user_id), nx=True, ex=ttl_seconds)

    if not acquired:
        holder = r.get(lock_key)
        print(f"🔒 Lock FAILED — ticket {ticket_id} already held by user {holder}")
        return None

    # Step 2: Create booking in PostgreSQL
    conn = get_pg_conn(autocommit=False)
    cur = conn.cursor()
    try:
        # Get ticket price
        cur.execute("SELECT price FROM tickets WHERE id = %s AND status = 'available'", (ticket_id,))
        row = cur.fetchone()
        if not row:
            conn.rollback()
            r.delete(lock_key)  # release Redis lock
            print(f"❌ Ticket {ticket_id} is not available in the database")
            return None

        price = row[0]

        # Create the booking with status 'in-progress'
        cur.execute(
            "INSERT INTO bookings (user_id, event_id, total_price, status) "
            "VALUES (%s, %s, %s, 'in-progress') RETURNING id",
            (user_id, event_id, price)
        )
        booking_id = cur.fetchone()[0]

        # Link ticket to booking
        cur.execute(
            "INSERT INTO booking_tickets (booking_id, ticket_id) VALUES (%s, %s)",
            (booking_id, ticket_id)
        )

        # Also store the booking_id in Redis for easy lookup
        r.set(f"ticket:{ticket_id}:booking", str(booking_id), ex=ttl_seconds)

        # Track reserved tickets per event (for seat map queries)
        r.sadd(f"event:{event_id}:reserved", str(ticket_id))

        conn.commit()
        print(f"✅ Reserved! ticket={ticket_id}, user={user_id}, booking={booking_id}, TTL={ttl_seconds}s")
        return booking_id

    except Exception as e:
        conn.rollback()
        r.delete(lock_key)
        print(f"❌ Error: {e}")
        return None
    finally:
        cur.close()
        conn.close()


# ── Demo: Two users fight over the same ticket ────────────
print("=" * 60)
print("🎬 Demo: Two users try to reserve the same ticket")
print("=" * 60)

# Pick an available ticket
cols, rows = query(
    "SELECT id, section, row_label, seat_number, price "
    "FROM tickets WHERE event_id = 1 AND status = 'available' "
    "ORDER BY id LIMIT 1"
)
ticket = rows[0]
tid = ticket[0]
print(f"\n🎫 Target: Ticket {tid} ({ticket[1]}-{ticket[2]}-{ticket[3]}, ${ticket[4]})\n")

# User 101 reserves first
booking_a = reserve_ticket(event_id=1, user_id=101, ticket_id=tid, ttl_seconds=600)

# User 102 tries the same ticket
booking_b = reserve_ticket(event_id=1, user_id=102, ticket_id=tid, ttl_seconds=600)

# Show the Redis key
r = get_redis()
lock_val = r.get(f"ticket:{tid}:lock")
lock_ttl = r.ttl(f"ticket:{tid}:lock")
print(f"\n🔍 Redis key 'ticket:{tid}:lock' = {lock_val} (TTL: {lock_ttl}s)")
print(f"\n📊 Result: User 101 → booking {booking_a} | User 102 → {booking_b}")

## 🔍 Checking Reservation Status

When rendering the seat map, we need to show reserved seats as **unavailable**
(even though they're still `available` in PostgreSQL). We use two strategies:

1. **Per-event set** — `event:{id}:reserved` is a Redis Set containing all
   currently reserved ticket IDs for an event. Fast O(1) membership check.

2. **Per-ticket TTL** — `ticket:{id}:lock` tells us how much time remains
   on the reservation.

This way the seat map can be rendered from a single Redis call instead of
scanning the database.

In [ ]:
# ── Query reservation status ────────────────────────

def get_reserved_tickets(event_id):
    """
    Return a list of ticket IDs that are currently reserved for an event.
    Uses the Redis set event:{id}:reserved, then filters out any whose
    lock has already expired (cleanup stale entries).
    """
    r = get_redis()
    set_key = f"event:{event_id}:reserved"
    members = r.smembers(set_key)

    active = []
    expired = []
    for tid_str in members:
        lock_key = f"ticket:{tid_str}:lock"
        if r.exists(lock_key):
            active.append(int(tid_str))
        else:
            # Lock expired — clean up the set
            expired.append(tid_str)

    # Remove expired entries from the set
    if expired:
        r.srem(set_key, *expired)

    return sorted(active)


def get_reservation_ttl(ticket_id):
    """
    Return (holder_user_id, remaining_seconds) for a reserved ticket.
    Returns (None, 0) if the ticket is not reserved.
    """
    r = get_redis()
    lock_key = f"ticket:{ticket_id}:lock"
    holder = r.get(lock_key)
    if holder is None:
        return None, 0
    ttl = r.ttl(lock_key)
    return int(holder), max(ttl, 0)


# ── Demo ──────────────────────────────────────────────
print("=" * 60)
print("🔍 Reservation Status Check")
print("=" * 60)

reserved = get_reserved_tickets(event_id=1)
print(f"\n🎫 Currently reserved tickets for Event 1: {reserved}")

if reserved:
    for tid in reserved:
        holder, ttl = get_reservation_ttl(tid)
        print(f"   ticket {tid}: held by user {holder}, {ttl}s remaining")

print(f"\n💡 The seat map would show these {len(reserved)} seat(s) as unavailable.")

## 💳 Simulating Payment

In production the payment flow looks like this:

```
1. Client tokenizes card via Stripe.js  →  token (tok_xxx)
2. Client sends token + bookingId to server
3. Server creates a Stripe PaymentIntent
4. Stripe processes the charge (2-30 seconds)
5. Stripe sends a webhook to confirm/decline
6. Server updates booking + ticket status
```

We'll simulate step 4 with a `sleep()` and a random success/failure (90% success).

The important part is step 6 — **confirming the booking**. This must be:

- **Guarded** — verify the Redis lock is still held by *this* user
- **Atomic** — use a PostgreSQL transaction to update ticket + booking together
- **Idempotent** — calling it twice with the same `booking_id` is a safe no-op

In [ ]:
# ── Simulate payment processing ─────────────────────

def simulate_payment(booking_id, delay_seconds=2):
    """
    Simulate Stripe payment processing.
    Returns True (paid) 90% of the time, False (declined) 10%.
    """
    print(f"💳 Processing payment for booking {booking_id} …")
    time.sleep(delay_seconds)
    success = random.random() < 0.9
    if success:
        print(f"✅ Payment SUCCEEDED for booking {booking_id}")
    else:
        print(f"❌ Payment DECLINED for booking {booking_id}")
    return success


def confirm_booking(booking_id, user_id, ticket_ids):
    """
    Confirm a booking after successful payment.

    Steps:
      1. Verify Redis lock is still held by this user
      2. PostgreSQL transaction: mark tickets 'sold', booking 'confirmed'
      3. Release Redis lock

    Idempotent: if booking is already 'confirmed', returns True immediately.

    Returns True on success, False on failure.
    """
    r = get_redis()
    conn = get_pg_conn(autocommit=False)
    cur = conn.cursor()

    try:
        # Idempotency check
        cur.execute("SELECT status, event_id FROM bookings WHERE id = %s", (booking_id,))
        row = cur.fetchone()
        if row is None:
            print(f"❌ Booking {booking_id} not found")
            conn.rollback()
            return False

        booking_status, event_id = row

        if booking_status == "confirmed":
            print(f"✅ Booking {booking_id} already confirmed (idempotent no-op)")
            conn.rollback()
            return True

        if booking_status == "cancelled":
            print(f"❌ Booking {booking_id} was already cancelled")
            conn.rollback()
            return False

        # Verify Redis locks
        for tid in ticket_ids:
            lock_key = f"ticket:{tid}:lock"
            holder = r.get(lock_key)
            if holder is None:
                print(f"⏰ Lock expired for ticket {tid} — reservation timed out!")
                cur.execute("UPDATE bookings SET status = 'cancelled' WHERE id = %s", (booking_id,))
                conn.commit()
                return False
            if int(holder) != user_id:
                print(f"🚫 Ticket {tid} is now held by user {holder}, not {user_id}")
                cur.execute("UPDATE bookings SET status = 'cancelled' WHERE id = %s", (booking_id,))
                conn.commit()
                return False

        # Update tickets to 'sold'
        # The WHERE status='available' clause is a safety net:
        # if somehow the ticket was already sold, this UPDATE
        # affects 0 rows and we detect the conflict.
        placeholders = ",".join(["%s"] * len(ticket_ids))
        cur.execute(
            f"UPDATE tickets SET status = 'sold' "
            f"WHERE id IN ({placeholders}) AND status = 'available' "
            f"RETURNING id",
            ticket_ids
        )
        sold_ids = [row[0] for row in cur.fetchall()]

        if len(sold_ids) != len(ticket_ids):
            print(f"⚠️  Only {len(sold_ids)}/{len(ticket_ids)} tickets were available — rolling back")
            conn.rollback()
            return False

        # Update booking to 'confirmed'
        cur.execute(
            "UPDATE bookings SET status = 'confirmed' WHERE id = %s",
            (booking_id,)
        )

        conn.commit()

        # Release Redis locks
        for tid in ticket_ids:
            r.delete(f"ticket:{tid}:lock")
            r.delete(f"ticket:{tid}:booking")
        r.srem(f"event:{event_id}:reserved", *[str(t) for t in ticket_ids])

        print(f"✅ Booking {booking_id} CONFIRMED — {len(sold_ids)} ticket(s) sold!")
        return True

    except Exception as e:
        conn.rollback()
        print(f"❌ Error confirming booking {booking_id}: {e}")
        return False
    finally:
        cur.close()
        conn.close()


def cancel_booking(booking_id, user_id, ticket_ids, event_id):
    """Cancel a booking and release all associated locks."""
    r = get_redis()
    conn = get_pg_conn()
    cur = conn.cursor()

    cur.execute("UPDATE bookings SET status = 'cancelled' WHERE id = %s", (booking_id,))

    for tid in ticket_ids:
        lock_key = f"ticket:{tid}:lock"
        holder = r.get(lock_key)
        if holder and int(holder) == user_id:
            r.delete(lock_key)
            r.delete(f"ticket:{tid}:booking")
    r.srem(f"event:{event_id}:reserved", *[str(t) for t in ticket_ids])

    cur.close()
    conn.close()
    print(f"🗑️  Booking {booking_id} cancelled, locks released")


# ── Demo: Full reserve → pay → confirm ─────────────────
print("=" * 60)
print("🎬 Demo: Reserve → Pay → Confirm")
print("=" * 60)

# Pick a fresh available ticket
cols, rows = query(
    "SELECT id, section, row_label, seat_number, price "
    "FROM tickets WHERE event_id = 1 AND status = 'available' "
    "ORDER BY id LIMIT 1 OFFSET 1"
)
ticket = rows[0]
demo_tid = ticket[0]
print(f"\n🎫 Ticket: {demo_tid} ({ticket[1]}-{ticket[2]}-{ticket[3]}, ${ticket[4]})")

# Step 1: Reserve
print("\n── Step 1: Reserve ──")
booking_id = reserve_ticket(event_id=1, user_id=201, ticket_id=demo_tid, ttl_seconds=600)

# Step 2: Pay
print("\n── Step 2: Pay ──")
paid = simulate_payment(booking_id, delay_seconds=1)

# Step 3: Confirm
if paid:
    print("\n── Step 3: Confirm ──")
    confirm_booking(booking_id, user_id=201, ticket_ids=[demo_tid])

# Verify in database
print("\n── Verification ──")
show(
    "SELECT t.id, t.section, t.row_label, t.seat_number, t.status, b.id as booking_id, b.status as booking_status "
    "FROM tickets t "
    "LEFT JOIN booking_tickets bt ON t.id = bt.ticket_id "
    "LEFT JOIN bookings b ON bt.booking_id = b.id "
    "WHERE t.id = %s",
    (demo_tid,),
    title="📊 Ticket + Booking status after confirmation:"
)

## ⏰ Handling TTL Expiration

What if the user gets distracted and never completes payment?

1. The Redis lock **expires automatically** when the TTL runs out
2. The ticket becomes available for other users immediately
3. The in-progress booking becomes **stale** — we cancel it on next access

No cron job. No background worker. Redis handles it for free.

Let's see this in action with a very short TTL (5 seconds).

In [ ]:
# ── Demo: TTL expiration ────────────────────────────
print("=" * 60)
print("🎬 Demo: TTL Expiration (5-second reservation)")
print("=" * 60)

# Pick another available ticket
cols, rows = query(
    "SELECT id, section, row_label, seat_number "
    "FROM tickets WHERE event_id = 1 AND status = 'available' "
    "ORDER BY id LIMIT 1 OFFSET 2"
)
ttl_ticket_id = rows[0][0]
print(f"\n🎫 Using ticket {ttl_ticket_id} ({rows[0][1]}-{rows[0][2]}-{rows[0][3]})")

# User 301 reserves with a 5-second TTL
print("\n── t=0s: User 301 reserves with TTL=5s ──")
booking_301 = reserve_ticket(event_id=1, user_id=301, ticket_id=ttl_ticket_id, ttl_seconds=5)

# Check lock
r = get_redis()
holder, ttl = get_reservation_ttl(ttl_ticket_id)
print(f"   Lock holder: user {holder}, TTL: {ttl}s")

# Wait for expiration
print("\n── Waiting 6 seconds for TTL to expire … ──")
time.sleep(6)

# Check lock again
holder, ttl = get_reservation_ttl(ttl_ticket_id)
print(f"\n── t=6s: Lock check ──")
print(f"   Lock holder: {holder}, TTL: {ttl}s")
print(f"   🔓 Lock has expired! Ticket is free again.")

# Another user can grab it now
print("\n── t=6s: User 302 tries to reserve the same ticket ──")
booking_302 = reserve_ticket(event_id=1, user_id=302, ticket_id=ttl_ticket_id, ttl_seconds=600)

# Original user tries to confirm — fails!
print("\n── t=6s: User 301 finally tries to confirm (too late!) ──")
result = confirm_booking(booking_301, user_id=301, ticket_ids=[ttl_ticket_id])
print(f"   Confirm result: {result}")

# Show the stale booking was cancelled
show(
    "SELECT id, user_id, status FROM bookings WHERE id = %s",
    (booking_301,),
    title="\n📊 User 301's stale booking:"
)

# Clean up: cancel booking_302 so the ticket is available for later demos
r = get_redis()
r.delete(f"ticket:{ttl_ticket_id}:lock")
r.delete(f"ticket:{ttl_ticket_id}:booking")
r.srem("event:1:reserved", str(ttl_ticket_id))
conn = get_pg_conn()
cur = conn.cursor()
cur.execute("UPDATE bookings SET status = 'cancelled' WHERE id = %s", (booking_302,))
cur.close()
conn.close()

## 🧊 Edge Case: TTL Expires During Payment

Here's a tricky scenario:

```
t=0s    User A reserves ticket (TTL = 10s)
t=8s    User A submits payment to Stripe
t=10s   Redis lock EXPIRES <- uh oh!
t=10.5s User B reserves the same ticket
t=11s   Stripe confirms User A's payment
t=11s   User A tries to confirm booking…
```

Both users think they own the ticket! How do we handle this?

### Two safety nets:

1. **PostgreSQL `WHERE status='available'`** — When confirming, we do:
   ```sql
   UPDATE tickets SET status='sold' WHERE id=42 AND status='available'
   ```
   Only one of the two confirmations will find the ticket `available`.
   The other gets 0 rows affected → we detect the conflict and **refund** the loser.

2. **Lock extension** — When payment begins, we **extend** the TTL to give
   Stripe enough time:
   ```python
   redis.expire(f"ticket:{id}:lock", new_ttl)
   ```
   This is much better because it avoids the refund scenario entirely.

In [ ]:
# ── Lock extension ──────────────────────────────────

def extend_reservation(ticket_id, user_id, extra_seconds=300):
    """
    Extend the TTL on a ticket reservation.

    Only the current holder can extend. This is called when the user
    submits payment, giving Stripe extra time to process.

    Returns True if extended, False if the lock is gone or held by someone else.
    """
    r = get_redis()
    lock_key = f"ticket:{ticket_id}:lock"

    # Check who holds the lock
    holder = r.get(lock_key)
    if holder is None:
        print(f"⏰ Cannot extend — lock for ticket {ticket_id} already expired")
        return False
    if int(holder) != user_id:
        print(f"🚫 Cannot extend — ticket {ticket_id} held by user {holder}, not {user_id}")
        return False

    # Extend the TTL
    current_ttl = r.ttl(lock_key)
    new_ttl = current_ttl + extra_seconds
    r.expire(lock_key, new_ttl)

    # Also extend the booking key
    booking_key = f"ticket:{ticket_id}:booking"
    if r.exists(booking_key):
        r.expire(booking_key, new_ttl)

    print(f"⏱️  Extended ticket {ticket_id} lock: {current_ttl}s → {new_ttl}s (+{extra_seconds}s)")
    return True


# ── Demo: Lock extension ────────────────────────────
print("=" * 60)
print("🎬 Demo: Extending a Reservation Lock")
print("=" * 60)

# Pick a fresh ticket
cols, rows = query(
    "SELECT id, section, row_label, seat_number "
    "FROM tickets WHERE event_id = 1 AND status = 'available' "
    "ORDER BY id LIMIT 1 OFFSET 3"
)
ext_ticket_id = rows[0][0]
print(f"\n🎫 Using ticket {ext_ticket_id} ({rows[0][1]}-{rows[0][2]}-{rows[0][3]})")

# Reserve with 10-second TTL
print("\n── Reserve with TTL=10s ──")
ext_booking = reserve_ticket(event_id=1, user_id=401, ticket_id=ext_ticket_id, ttl_seconds=10)

# Check TTL
_, ttl_before = get_reservation_ttl(ext_ticket_id)
print(f"   TTL before extension: {ttl_before}s")

# Wait 3 seconds (simulating user filling form)
time.sleep(3)
_, ttl_mid = get_reservation_ttl(ext_ticket_id)
print(f"\n   After 3s: TTL = {ttl_mid}s")

# User clicks "Pay" -> extend the lock
print("\n── User submits payment → extend lock by 300s ──")
extend_reservation(ext_ticket_id, user_id=401, extra_seconds=300)

_, ttl_after = get_reservation_ttl(ext_ticket_id)
print(f"   TTL after extension: {ttl_after}s  ✅ Plenty of time for Stripe!")

# Clean up: confirm this booking
confirm_booking(ext_booking, user_id=401, ticket_ids=[ext_ticket_id])

## 🏁 Full Reserve → Pay → Confirm Flow

Let's put it all together in one clean simulation. We'll run **5 users**
simultaneously, each trying to book a ticket for Event 1:

| User | Scenario |
|------|----------|
| User 501 | ✅ Normal flow — reserve, pay, confirm |
| User 502 | ✅ Normal flow — different ticket |
| User 503 | ❌ Payment **declined** by Stripe |
| User 504 | ⏰ TTL **expires** before payment |
| User 505 | ✅ Grabs the ticket freed by User 504's expiration |

This mirrors what happens in a real system during a flash sale.

In [ ]:
# ── Full end-to-end simulation ──────────────────────
print("=" * 60)
print("🎬 Full End-to-End Simulation: 5 Users")
print("=" * 60)

# Get 4 available tickets for event 1
cols, rows = query(
    "SELECT id, section, row_label, seat_number, price "
    "FROM tickets WHERE event_id = 1 AND status = 'available' "
    "ORDER BY id LIMIT 4 OFFSET 4"
)
available_tickets = rows
print(f"\n🎫 Available tickets for this simulation:")
print(tabulate(available_tickets, headers=cols, tablefmt="simple_outline"))

t1, t2, t3, t4 = [r[0] for r in available_tickets]
timeline = []
t0 = time.time()

def log(msg):
    elapsed = time.time() - t0
    timestamp = f"t={elapsed:5.1f}s"
    timeline.append((timestamp, msg))
    print(f"  [{timestamp}] {msg}")


# ── User 501: Normal successful flow ──────────────────
print("\n── User 501: Happy path ──")
b501 = reserve_ticket(event_id=1, user_id=501, ticket_id=t1, ttl_seconds=600)
log(f"User 501 reserved ticket {t1} → booking {b501}")
time.sleep(0.5)
paid = simulate_payment(b501, delay_seconds=1)
log(f"User 501 payment: {"✅ success" if paid else "❌ failed"}")
if paid:
    confirm_booking(b501, user_id=501, ticket_ids=[t1])
    log(f"User 501 confirmed booking {b501} ✅")

# ── User 502: Normal successful flow ──────────────────
print("\n── User 502: Happy path ──")
b502 = reserve_ticket(event_id=1, user_id=502, ticket_id=t2, ttl_seconds=600)
log(f"User 502 reserved ticket {t2} → booking {b502}")
time.sleep(0.5)
paid = simulate_payment(b502, delay_seconds=1)
log(f"User 502 payment: {"✅ success" if paid else "❌ failed"}")
if paid:
    confirm_booking(b502, user_id=502, ticket_ids=[t2])
    log(f"User 502 confirmed booking {b502} ✅")

# ── User 503: Payment fails ──────────────────────────
print("\n── User 503: Payment declined ──")
b503 = reserve_ticket(event_id=1, user_id=503, ticket_id=t3, ttl_seconds=600)
log(f"User 503 reserved ticket {t3} → booking {b503}")
time.sleep(0.5)
# Force payment failure for demo
print(f"💳 Processing payment for booking {b503} …")
time.sleep(1)
print(f"❌ Payment DECLINED for booking {b503}")
log(f"User 503 payment: ❌ declined")

# Payment failed -> cancel booking and release lock
cancel_booking(b503, user_id=503, ticket_ids=[t3], event_id=1)
log(f"User 503 booking cancelled, ticket {t3} released 🔓")

# ── User 504: TTL expires ────────────────────────────
print("\n── User 504: TTL expires (3-second TTL) ──")
b504 = reserve_ticket(event_id=1, user_id=504, ticket_id=t4, ttl_seconds=3)
log(f"User 504 reserved ticket {t4} → booking {b504} (TTL=3s)")

print("   ⏳ User 504 is distracted … waiting 4 seconds …")
time.sleep(4)
log(f"User 504 TTL expired — ticket {t4} auto-released 🔓")

# ── User 505: Grabs the freed ticket ─────────────────
print("\n── User 505: Grabs ticket freed by User 504 ──")
b505 = reserve_ticket(event_id=1, user_id=505, ticket_id=t4, ttl_seconds=600)
log(f"User 505 reserved ticket {t4} → booking {b505}")
time.sleep(0.5)
paid = simulate_payment(b505, delay_seconds=1)
log(f"User 505 payment: {"✅ success" if paid else "❌ failed"}")
if paid:
    confirm_booking(b505, user_id=505, ticket_ids=[t4])
    log(f"User 505 confirmed booking {b505} ✅")

# User 504 finally tries to confirm — too late!
print("\n── User 504 tries to confirm (too late!) ──")
result = confirm_booking(b504, user_id=504, ticket_ids=[t4])
log(f"User 504 confirm attempt: {"✅" if result else "❌ FAILED (lock expired)"}")

# ── Print timeline ──────────────────────────────────
print("\n" + "=" * 60)
print("📋 Event Timeline")
print("=" * 60)
for ts, msg in timeline:
    print(f"  {ts}  │  {msg}")

# ── Final state ────────────────────────────────────
print("\n" + "=" * 60)
print("📊 Final State")
print("=" * 60)
show(
    "SELECT t.id, t.section, t.row_label, t.seat_number, t.price, t.status as ticket_status, "
    "       b.id as booking_id, b.user_id, b.status as booking_status "
    "FROM tickets t "
    "LEFT JOIN booking_tickets bt ON t.id = bt.ticket_id "
    "LEFT JOIN bookings b ON bt.booking_id = b.id "
    "WHERE t.id IN %s "
    "ORDER BY t.id",
    ((t1, t2, t3, t4),),
    title="Tickets used in simulation:"
)

## 🧹 Cleanup

Run this cell to reset all changes made during this notebook:
- Delete Redis lock/booking/set keys
- Reset ticket statuses back to `available`
- Remove test bookings

In [ ]:
# ── Cleanup ─────────────────────────────────────────
print("🧹 Cleaning up …\n")

r = get_redis()

# 1. Delete all notebook Redis keys
cursor = 0
deleted_keys = 0
while True:
    cursor, keys = r.scan(cursor, match="ticket:*", count=100)
    if keys:
        r.delete(*keys)
        deleted_keys += len(keys)
    if cursor == 0:
        break

cursor = 0
while True:
    cursor, keys = r.scan(cursor, match="event:*:reserved", count=100)
    if keys:
        r.delete(*keys)
        deleted_keys += len(keys)
    if cursor == 0:
        break

print(f"   🔑 Deleted {deleted_keys} Redis keys")

# 2. Reset tickets: any ticket marked 'sold' by our test users -> 'available'
conn = get_pg_conn()
cur = conn.cursor()

# Find tickets linked to our test bookings (user_ids 101-599)
cur.execute(
    "SELECT bt.ticket_id FROM booking_tickets bt "
    "JOIN bookings b ON bt.booking_id = b.id "
    "WHERE b.user_id BETWEEN 101 AND 599"
)
test_ticket_ids = [row[0] for row in cur.fetchall()]

if test_ticket_ids:
    placeholders = ",".join(["%s"] * len(test_ticket_ids))
    cur.execute(
        f"UPDATE tickets SET status = 'available' WHERE id IN ({placeholders}) AND status = 'sold'",
        test_ticket_ids
    )
    print(f"   🎫 Reset {cur.rowcount} ticket(s) to 'available'")

# 3. Delete test booking_tickets and bookings
cur.execute(
    "DELETE FROM booking_tickets WHERE booking_id IN "
    "(SELECT id FROM bookings WHERE user_id BETWEEN 101 AND 599)"
)
bt_count = cur.rowcount

cur.execute("DELETE FROM bookings WHERE user_id BETWEEN 101 AND 599")
b_count = cur.rowcount

print(f"   📦 Deleted {b_count} test booking(s) and {bt_count} booking_ticket link(s)")

cur.close()
conn.close()

print("\n✅ Cleanup complete! Database and Redis are back to their original state.")

## 📝 Summary

### The Complete Payment Flow

```
┌─────────┐     ┌────────────────────────┐     ┌────────────┐     ┌────────────┐
│  User    │     │   Booking Service      │     │   Redis    │     │ PostgreSQL │
│  clicks  │     │                        │     │            │     │            │
│  "Buy"   │────▶│ 1. SET NX EX (lock)   │────▶│ ✅ locked  │     │            │
│          │     │ 2. INSERT booking      │──────────────────▶│ in-progress│
│          │     │                        │     │            │     │            │
│  enters  │     │                        │     │            │     │            │
│  card    │     │ 3. EXPIRE (extend TTL) │────▶│ ⏱️ extended │     │            │
│          │     │                        │     │            │     │            │
│  clicks  │     │ 4. Charge via Stripe   │     │            │     │            │
│  "Pay"   │     │                        │     │            │     │            │
│          │     │ 5. UPDATE ticket=sold  │──────────────────▶│ sold ✅    │
│          │     │    UPDATE booking=conf │──────────────────▶│ confirmed  │
│          │     │ 6. DEL lock            │────▶│ 🔓 freed   │     │            │
└─────────┘     └────────────────────────┘     └────────────┘     └────────────┘

If user abandons:
  Redis TTL expires → lock auto-deleted → ticket free for others

If payment fails:
  DEL lock → cancel booking → ticket free for others
```

### 🔑 Key Takeaways

| Approach | Verdict | Why |
|----------|---------|-----|
| Long DB lock (`FOR UPDATE`) | ❌ Never | Blocks connections, deadlocks, doesn't scale |
| DB status + cron | ⚠️ OK-ish | Works but has cron lag and clock drift issues |
| **Redis distributed lock** | ✅ Best | Atomic, self-cleaning TTL, 100K+ ops/sec |

### 🧠 Concepts to Remember

1. **`SET NX EX`** is an atomic "try-lock with timeout" — the perfect primitive
   for temporary reservations.

2. **TTL handles abandonment** — no cron jobs, no background workers. Redis
   cleans up automatically.

3. **Idempotent confirmation** — checking `if booking.status == 'confirmed'`
   before doing work means duplicate webhooks are harmless.

4. **PostgreSQL as safety net** — `WHERE status='available'` in the UPDATE
   prevents double-selling even if Redis fails.

5. **Lock extension** — when payment begins, extend the TTL so Stripe has
   enough time to process.

---

### 🔜 Next Up: Notebook 4 — Scaling Ticket Inventory

How do you serve **millions** of users checking seat availability without
melting the database? We'll explore **caching patterns**, **read-through
caches**, and **cache invalidation** strategies.